# Gravimetry Field QC Viewer
**Lunar Leaper / Scintrex CG-5**

Inspect repeated readings per station, apply flagging thresholds, and manually exclude individual readings before computing station averages.

In [14]:
# ── Dependencies ────────────────────────────────────────────────────────────
# Run this cell once to install required packages
# !pip install plotly ipywidgets pandas

In [15]:
# ── Imports ───────────────────────────────────────────────────────────────
import re
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
widgets.Widget.close_all()  # clear any stale widget state
from pathlib import Path

## 1. Load data
Set `DATA_FILE` to the `.txt` export from the CG-5. Re-run this cell after transferring new data.

In [16]:
# ── Set your data file path here ────────────────────────────────────────────
DATA_FILE = Path("../../Data/Gravimetry/Field data/2014May012026.txt")  # <-- change this

In [17]:
# ── Parser ───────────────────────────────────────────────────────────────────
def parse_cg5(filepath):
    col_names = [
        "Line", "Station", "Alt", "Grav", "SD",
        "TiltX", "TiltY", "Temp", "Tide", "Dur", "Rej",
        "Time", "DecTime", "Terrain", "Date"
    ]

    records = []
    current_survey = None

    with open(filepath, "r") as f:
        for raw_line in f:
            line = raw_line.strip()

            m = re.match(r"/\s*Survey name:\s*(\S+)", line)
            if m:
                current_survey = m.group(1)
                continue

            if line.startswith("/") or line.startswith("Line") or not line:
                continue

            parts = line.split()
            if len(parts) >= 15:
                try:
                    row = dict(zip(col_names, parts))  # <-- fixed
                    row["Survey"] = current_survey
                    row["Datetime"] = pd.to_datetime(
                        row["Date"] + " " + row["Time"],
                        format="%Y/%m/%d %H:%M:%S"
                    )
                    records.append(row)
                except (ValueError, KeyError):
                    pass

    df = pd.DataFrame(records)

    for col in ["Line", "Station", "Grav", "SD", "TiltX", "TiltY",
                "Temp", "Tide", "Dur", "Rej"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["Station"] = df["Station"].astype(int)

    return df.sort_values("Datetime").reset_index(drop=True)


df = parse_cg5(DATA_FILE)

print(f"Loaded {len(df)} readings across {df['Station'].nunique()} stations "
      f"from {df['Survey'].nunique()} survey(s): {sorted(df['Survey'].unique())}")
df.head()

Loaded 1392 readings across 39 stations from 2 survey(s): ['danza', 'kanza']


,Line,Station,Alt,Grav,SD,TiltX,TiltY,Temp,Tide,Dur,Rej,Time,DecTime,Terrain,Date,Survey,Datetime
0,0.0,1000,26.1717,6653.462,0.053,-9.6,-5.2,286.84,0.143,30,0,15:36:05,46075.64902,0.0000,2026/03/23,kanza,2026-03-23 15:36:05
1,0.0,1000,26.1717,6653.462,0.048,-10.6,-6.2,286.84,0.143,30,0,15:36:40,46075.64942,0.0000,2026/03/23,kanza,2026-03-23 15:36:40
2,0.0,1000,26.1717,6653.463,0.035,-11.0,-6.5,286.85,0.143,30,0,15:37:14,46075.64982,0.0000,2026/03/23,kanza,2026-03-23 15:37:14
3,0.0,1000,25.6834,6655.129,1.033,-11.0,-6.5,286.85,0.143,0,0,15:37:49,46075.65022,0.0000,2026/03/23,kanza,2026-03-23 15:37:49
4,0.0,1000,26.6600,6653.424,0.044,151.8,-1.3,286.84,0.144,30,0,15:43:09,46075.65392,0.0000,2026/03/23,kanza,2026-03-23 15:43:09


## 2. Per-station QC viewer

Select a line and station, then set the flagging thresholds below. A reading is flagged (red) if any threshold is exceeded. Flagged readings are excluded from the mean and SE but are not deleted.

Use the deselect box to manually exclude specific readings regardless of the thresholds. The tilt threshold applies to each axis independently: a reading is flagged if |TiltX| or |TiltY| exceeds the limit.

The summary below the plot shows the weighted mean and instrument SE for the unflagged readings. Scatter-based SE is not reported at this stage because consecutive readings contain instrument drift; it will be reintroduced after loop correction.

In [18]:
# ── Widget definitions ───────────────────────────────────────────────────────
lines_sorted = sorted(df["Line"].unique())

w_line = widgets.Dropdown(
    options=lines_sorted,
    description="Line:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px")
)

def stations_for_line(line):
    return sorted(df[df["Line"] == line]["Station"].unique())

w_station = widgets.Dropdown(
    options=stations_for_line(lines_sorted[0]),
    description="Station:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="220px")
)

def on_line_change(change):
    w_station.options = stations_for_line(change["new"])

w_line.observe(on_line_change, names="value")

w_sd = widgets.FloatSlider(
    value=0.10, min=0.01, max=10.0, step=0.01,
    description="SD threshold:",
    continuous_update=False,
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px")
)

w_rej = widgets.IntSlider(
    value=10, min=0, max=60, step=1,
    description="Max rejections:",
    continuous_update=False,
    style={"description_width": "120px"},
    layout=widgets.Layout(width="420px")
)

w_tilt = widgets.FloatSlider(
    value=20.0, min=0, max=100, step=1,
    description="Max |tilt| (X or Y):",
    continuous_update=False,
    style={"description_width": "140px"},
    layout=widgets.Layout(width="440px")
)

w_deselect = widgets.SelectMultiple(
    options=[],
    description="Deselect:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="320px", height="120px")
)
w_deselect_label = widgets.HTML(
    "<small>Hold Ctrl to select multiple readings to exclude from plot & mean</small>"
)

out = widgets.Output()

# ── Flag reason helper ───────────────────────────────────────────────────────
def flag_reasons(row, sd_thresh, rej_thresh, tilt_thresh):
    reasons = []
    if row["SD"] > sd_thresh:
        reasons.append(f"SD={row['SD']:.3f}>{sd_thresh}")
    if row["Rej"] > rej_thresh:
        reasons.append(f"Rej={int(row['Rej'])}>{rej_thresh}")
    if abs(row["TiltX"]) > tilt_thresh:
        reasons.append(f"TiltX={row['TiltX']:.1f}>{tilt_thresh}")
    if abs(row["TiltY"]) > tilt_thresh:
        reasons.append(f"TiltY={row['TiltY']:.1f}>{tilt_thresh}")
    return ", ".join(reasons) if reasons else ""

# ── Weighted stats helper ────────────────────────────────────────────────────
def weighted_stats(grp):
    """
    Weighted mean and SE based on internal instrument noise only.

    Each reading is the mean of (Dur * 6 - Rej) accepted 6 Hz samples, so its SE is:
        SE_i = SD_i / sqrt(Dur_i * 6 - Rej_i)

    Readings are combined with weights w_i = 1 / SE_i^2:
        g_w = sum(w_i * g_i) / sum(w_i)
        SE  = 1 / sqrt(sum(w_i))

    Scatter-based SE is not included here: consecutive readings contain
    instrument drift that inflates the scatter until loop correction is applied.
    """
    n_samples = grp["Dur"] - grp["Rej"]
    SE_i      = grp["SD"] / np.sqrt(n_samples)
    w_i       = 1.0 / SE_i**2
    g_w       = (w_i * grp["Grav"]).sum() / w_i.sum()
    SE_sta    = 1.0 / np.sqrt(w_i.sum())
    return g_w, SE_sta

# ── Plotting function ────────────────────────────────────────────────────────
def plot_station(line, station, sd_thresh, rej_thresh, tilt_thresh, deselected):
    sub = df[(df["Line"] == line) & (df["Station"] == station)].copy()

    if sub.empty:
        print(f"No data for Line {line}, Station {station}")
        return

    sub["flag_reason"] = sub.apply(
        lambda r: flag_reasons(r, sd_thresh, rej_thresh, tilt_thresh), axis=1
    )
    sub["flagged"]    = sub["flag_reason"] != ""
    sub["deselected"] = sub["Time"].isin(deselected)

    time_labels = list(sub["Time"])
    if list(w_deselect.options) != time_labels:
        w_deselect.options = time_labels

    good     = sub[~sub["flagged"] & ~sub["deselected"]]
    bad      = sub[ sub["flagged"] & ~sub["deselected"]]
    excluded = sub[ sub["deselected"]]

    n_total    = len(sub)
    n_flagged  = len(bad)
    n_excluded = len(excluded)

    def hover(df_in, include_reason=False):
        rows = []
        for row in df_in.itertuples():
            h = (
                f"<b>Time:</b> {row.Time}<br>"
                f"<b>Grav:</b> {row.Grav:.4f} mGal<br>"
                f"<b>SD:</b> {row.SD:.4f}<br>"
                f"<b>Rej:</b> {int(row.Rej)}<br>"
                f"<b>TiltX:</b> {row.TiltX:.1f}  TiltY: {row.TiltY:.1f}<br>"
                f"<b>Dur:</b> {int(row.Dur)} s<br>"
                f"<b>Survey:</b> {row.Survey}"
            )
            if include_reason and row.flag_reason:
                h += f"<br><b>Flagged:</b> {row.flag_reason}"
            rows.append(h)
        return rows

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        subplot_titles=("Gravity (mGal)", "Standard Deviation", "Tilt X / Tilt Y (arcsec)"),
        vertical_spacing=0.08,
        row_heights=[0.5, 0.25, 0.25]
    )

    # ── Row 1: Gravity ───────────────────────────────────────────────────────
    if not good.empty:
        fig.add_trace(go.Scatter(
            x=good["Datetime"], y=good["Grav"],
            mode="markers", name="Good",
            marker=dict(color="steelblue", size=9),
            text=hover(good), hovertemplate="%{text}<extra></extra>"
        ), row=1, col=1)

    if not bad.empty:
        fig.add_trace(go.Scatter(
            x=bad["Datetime"], y=bad["Grav"],
            mode="markers", name="Flagged",
            marker=dict(color="crimson", size=11, symbol="x"),
            text=hover(bad, include_reason=True),
            hovertemplate="%{text}<extra></extra>"
        ), row=1, col=1)

    if not excluded.empty:
        fig.add_trace(go.Scatter(
            x=excluded["Datetime"], y=excluded["Grav"],
            mode="markers", name="Deselected",
            marker=dict(color="gray", size=9, symbol="circle-open"),
            text=hover(excluded, include_reason=True),
            hovertemplate="%{text}<extra></extra>"
        ), row=1, col=1)

    if not good.empty:
        g_w, SE_sta = weighted_stats(good)
        fig.add_hline(
            y=g_w, line_dash="dot", line_color="steelblue",
            annotation_text=f"weighted mean: {g_w:.4f} mGal",
            annotation_position="top right", row=1, col=1
        )

    # ── Row 2: SD ────────────────────────────────────────────────────────────
    for df_part, color in [(good, "steelblue"), (bad, "crimson"), (excluded, "lightgray")]:
        if df_part.empty:
            continue
        fig.add_trace(go.Bar(
            x=df_part["Datetime"], y=df_part["SD"],
            marker_color=color, showlegend=False,
            hovertemplate="SD: %{y:.4f}<extra></extra>"
        ), row=2, col=1)

    fig.add_hline(
        y=sd_thresh, line_dash="dash", line_color="orange",
        annotation_text=f"SD limit: {sd_thresh}",
        annotation_position="top right", row=2, col=1
    )

    # ── Row 3: Tilt ──────────────────────────────────────────────────────────
    for df_part, color in [(good, "steelblue"), (bad, "crimson"), (excluded, "lightgray")]:
        if df_part.empty:
            continue
        fig.add_trace(go.Scatter(
            x=df_part["Datetime"], y=df_part["TiltX"],
            mode="markers", showlegend=False,
            marker=dict(color=color, size=7, symbol="circle"),
            hovertemplate="TiltX: %{y:.1f}<extra></extra>"
        ), row=3, col=1)
        fig.add_trace(go.Scatter(
            x=df_part["Datetime"], y=df_part["TiltY"],
            mode="markers", showlegend=False,
            marker=dict(color=color, size=7, symbol="square", opacity=0.6),
            hovertemplate="TiltY: %{y:.1f}<extra></extra>"
        ), row=3, col=1)

    fig.add_hline(y=0,            line_dash="dot",  line_color="gray",   row=3, col=1)
    fig.add_hline(y= tilt_thresh, line_dash="dash", line_color="orange", row=3, col=1)
    fig.add_hline(y=-tilt_thresh, line_dash="dash", line_color="orange", row=3, col=1)

    fig.update_layout(
        title=dict(
            text=(
                f"Line {line} / Station {station} / {n_total} readings  "
                f"<span style='color:crimson'>{n_flagged} flagged</span>  "
                f"<span style='color:gray'>{n_excluded} deselected</span>"
            ),
            font=dict(size=15)
        ),
        height=700,
        hovermode="closest",
        legend=dict(orientation="h", y=1.05, x=0),
        plot_bgcolor="#f9f9f9",
        paper_bgcolor="white",
        margin=dict(t=90, b=40)
    )
    fig.update_yaxes(title_text="mGal",   row=1, col=1, showgrid=True, gridcolor="#e0e0e0")
    fig.update_yaxes(title_text="SD",     row=2, col=1, showgrid=True, gridcolor="#e0e0e0")
    fig.update_yaxes(title_text="arcsec", row=3, col=1, showgrid=True, gridcolor="#e0e0e0")
    fig.update_xaxes(title_text="Time",   row=3, col=1, showgrid=True, gridcolor="#e0e0e0")
    fig.update_xaxes(showgrid=True, gridcolor="#e0e0e0")
    fig.show()

    # ── Summary ──────────────────────────────────────────────────────────────
    print(f"\nLine {line} / Station {station}")
    print(f"  Total readings : {n_total}")
    print(f"  Flagged        : {n_flagged}")
    print(f"  Deselected     : {n_excluded}")
    if not good.empty:
        n = len(good)
        g_w, SE_sta = weighted_stats(good)
        print(f"  Weighted mean  : {g_w:.4f} mGal  (n={n})")
        print(f"  SE instrument  : {SE_sta:.4f} mGal  [1/sqrt(sum(w_i)), w_i = (Dur - Rej)/SD_i^2]")
        print(f"  Range          : {good['Grav'].min():.4f} to {good['Grav'].max():.4f} mGal")
    else:
        print("  No good readings remain — check thresholds or deselection")
    if n_flagged > 0:
        print("\n  Flagged readings:")
        for row in bad.itertuples():
            print(f"    {row.Time}  ->  {row.flag_reason}")

# ── Wire up widgets ──────────────────────────────────────────────────────────
def on_change(_):
    out.clear_output(wait=True)
    with out:
        plot_station(
            w_line.value, w_station.value,
            w_sd.value, w_rej.value, w_tilt.value,
            list(w_deselect.value)
        )

for w in [w_line, w_station, w_sd, w_rej, w_tilt, w_deselect]:
    w.observe(on_change, names="value")

# ── Display ──────────────────────────────────────────────────────────────────
controls = widgets.VBox([
    widgets.HTML("<b>Selection</b>"),
    widgets.HBox([w_line, w_station]),
    widgets.HTML("<br><b>Flagging thresholds</b>"),
    w_sd, w_rej, w_tilt,
    widgets.HTML("<br><b>Manual exclusion</b>"),
    w_deselect_label,
    w_deselect
])

display(controls)
display(out)
on_change(None)

Output()

## 3. Overview by line

Mean gravity per station for each line, using the same thresholds set in section 2. Error bars show the combined standard error. Re-run this cell after changing thresholds above.

In [19]:
# Uses the last threshold values from the sliders above
# Re-run after adjusting thresholds in section 2

df["flagged"] = (
    (df["SD"]           > w_sd.value)   |
    (df["Rej"]          > w_rej.value)  |
    (df["TiltX"].abs()  > w_tilt.value) |
    (df["TiltY"].abs()  > w_tilt.value)
)

good_all = df[~df["flagged"]]

def station_summary(grp):
    g_w, SE_sta = weighted_stats(grp)
    return pd.Series({
        "n"       : len(grp),
        "mean"    : round(g_w,    4),
        "SE_instr": round(SE_sta, 4),
    })

summary = (
    good_all.groupby(["Line", "Station"])
    .apply(station_summary, include_groups=False)
    .reset_index()
)

all_lines = sorted(summary["Line"].unique())
colors = [
    "steelblue", "darkorange", "seagreen", "mediumpurple",
    "crimson", "saddlebrown", "teal", "goldenrod"
]

# ── Widget ───────────────────────────────────────────────────────────────────
w_lines_overview = widgets.SelectMultiple(
    options=all_lines,
    value=all_lines,
    description="Lines:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="220px", height=f"{min(120, 30 + 20 * len(all_lines))}px")
)
w_lines_label = widgets.HTML("<small>Hold Ctrl to select multiple lines</small>")
out2 = widgets.Output()

def plot_overview(selected_lines):
    sub_summary = summary[summary["Line"].isin(selected_lines)]

    fig2 = go.Figure()
    for i, line in enumerate(all_lines):
        if line not in selected_lines:
            continue
        sub = sub_summary[sub_summary["Line"] == line]
        color = colors[i % len(colors)]
        fig2.add_trace(go.Scatter(
            x=sub["Station"],
            y=sub["mean"],
            error_y=dict(type="data", array=sub["SE_instr"].values, visible=True),
            mode="markers+lines",
            name=f"Line {line}",
            marker=dict(size=9, color=color),
            line=dict(color=color, dash="dot"),
            hovertemplate=(
                f"<b>Line {line} / Station %{{x}}</b><br>"
                "Weighted mean: %{y:.4f} mGal<br>"
                "SE: +/-%{error_y.array:.4f} mGal<br>"
                "<extra></extra>"
            )
        ))

    fig2.update_layout(
        title="Weighted mean gravity per station by line (good readings, +/- SE instrument)",
        xaxis_title="Station",
        yaxis_title="Gravity (mGal)",
        plot_bgcolor="#f9f9f9",
        legend=dict(orientation="h", y=1.08, x=0),
        height=450
    )
    fig2.update_xaxes(showgrid=True, gridcolor="#e0e0e0")
    fig2.update_yaxes(showgrid=True, gridcolor="#e0e0e0")
    fig2.show()

    display(sub_summary[["Line", "Station", "n", "mean", "SE_instr"]])

def on_overview_change(_):
    out2.clear_output(wait=True)
    with out2:
        if not w_lines_overview.value:
            print("No lines selected.")
            return
        plot_overview(list(w_lines_overview.value))

w_lines_overview.observe(on_overview_change, names="value")

display(widgets.VBox([w_lines_label, w_lines_overview]))
display(out2)
on_overview_change(None)

Output()